# Lab 05 — Regressão linear com Scikit-Learn e SGD
**Curso 1 · Semana 2 · Machine Learning — Andrew Ng**

Adaptação de C1_W2_Lab05_Sklearn_GD_Soln.ipynb. O arquivo original usa load_house_data() dos módulos auxiliares do curso; nem os módulos nem o dataset foram anexados. Para que o caderno seja independente, usamos uma base **simulada** com as mesmas quatro características e unidades. Os pesos e resultados não reproduzem o dataset real. A sequência de operações do laboratório é preservada.

## 1. Por que usar Scikit-Learn?
Nos laboratórios anteriores implementamos custo, gradiente e atualização dos pesos. O Scikit-Learn oferece essas operações prontas, testadas e com uma interface padronizada. O objetivo agora é compreender o que as funções fazem, não apenas decorar fit e predict.

Neste laboratório o professor usa **SGDRegressor**, que utiliza descida do gradiente estocástica, e StandardScaler para Z-score. Não é o mesmo algoritmo de atualização em lote completo dos primeiros laboratórios.

Instalação, se necessário: python -m pip install numpy matplotlib scikit-learn.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler

np.set_printoptions(precision=2, suppress=True)

## 2. Carregamento e estrutura dos dados

O original chama load_house_data() para devolver X_train, y_train. X tem quatro colunas: área em ft², quartos, andares, idade em anos; y representa preços em milhares de dólares. Cada linha é uma casa. Na prática independente, geramos 100 casas artificiais com uma relação conhecida e ruído. A semente 42 torna os resultados reproduzíveis. A base não representa transações imobiliárias reais.

np.column_stack combina as quatro colunas. O preço simulado é a soma de intercepto, efeitos das características e ruído normal. As dimensões são (100,4) e (100,). O índice i escolhe uma casa e j uma característica.

In [ ]:
rng = np.random.default_rng(42)
m = 100
area = rng.uniform(300,2000,m)
quartos = rng.integers(1,6,m)
andares = rng.integers(1,3,m)
idade = rng.integers(0,80,m)
X_train = np.column_stack((area, quartos, andares, idade))
y_train = 90 + 0.20*area + 12*quartos + 15*andares - 1.2*idade
y_train += rng.normal(0,20,m)
X_features = ["Área (ft²)", "Quartos", "Andares", "Idade"]
print("X:", X_train.shape, "y:", y_train.shape)
print(X_train[:4])
print(y_train[:4])

## 3. StandardScaler: normalização pronta

No laboratório anterior implementamos Z-score:
$$x_j'=\frac{x_j-\mu_j}{\sigma_j}$$

StandardScaler() cria um objeto que armazena as estatísticas por coluna. Criar o objeto não transforma os dados ainda. fit(X_train) calcula médias e variâncias; transform(X_train) aplica as estatísticas já calculadas; fit_transform faz as duas etapas juntas.

mean_ guarda a média; var_ a variância; scale_ o fator de escala (desvio padrão, ou 1 para coluna constante). np.ptp calcula a amplitude. axis=0 significa operar por coluna. O Z-score não garante valores entre -1 e 1; ele centraliza e iguala a dispersão de colunas não constantes.

Ajuste o scaler apenas no treino quando houver conjuntos separados. Reutilize transform para validação, teste e novos exemplos; fit_transform neles provocaria vazamento de dados ou escalas incompatíveis.

In [ ]:
scaler = StandardScaler()
X_norm = scaler.fit_transform(X_train)
print("Amplitude original:", np.ptp(X_train,axis=0))
print("Amplitude normalizada:", np.ptp(X_norm,axis=0))
print("Médias:", scaler.mean_)
print("Variâncias:", scaler.var_)
print("Escalas:", scaler.scale_)
print("Médias depois:", X_norm.mean(axis=0))
print("Desvios depois:", X_norm.std(axis=0))
print("Equivalência:", np.allclose(X_norm, (X_train-scaler.mean_)/scaler.scale_))

## 4. Criar e treinar o SGDRegressor

O original executa SGDRegressor(max_iter=1000) e sgdr.fit(X_norm,y_train). Mantemos max_iter e acrescentamos random_state=42 para reprodução.

- SGDRegressor: estimador de regressão linear treinado por descida do gradiente estocástica.
- max_iter=1000: máximo de épocas (passagens pelos dados), não número fixo de atualizações. Pode parar antes por tolerância.
- fit: ajusta os parâmetros a partir de X e y; modifica o objeto treinado.
- n_iter_: número de épocas efetivamente realizadas.
- t_: contador interno da taxa de aprendizado, relacionado às atualizações (normalmente n_iter_ × m + 1 numa chamada fit sem early stopping). Não é simplesmente o número de épocas.
- random_state: controla aleatoriedade, como a ordem de exemplos.

### Batch x SGD
No batch gradient descent, calculamos a média das contribuições de todos os exemplos e só então atualizamos os parâmetros:
$$\frac{\partial J}{\partial w_j}=\frac{1}{m}\sum_i(\hat y^{(i)}-y^{(i)})x_j^{(i)}$$

No SGD, atualizamos os parâmetros usando um exemplo por vez. Para o erro quadrático, uma contribuição é $(\hat y-y)x_j$ para w e $\hat y-y$ para b. O estimador também pode adicionar regularização. A curva de custo pode oscilar, ao contrário do batch com taxa suficientemente pequena.

Os padrões usuais de SGDRegressor incluem perda squared_error, penalização L2, alpha=0.0001 (força da regularização, **não** a taxa de aprendizado da aula), eta0=0.01 (taxa inicial), learning_rate='invscaling' e tol=0.001. A taxa muda ao longo das atualizações no modo invscaling. Consulte get_params() para os valores exatos da versão instalada.

In [ ]:
sgdr = SGDRegressor(max_iter=1000, random_state=42)
sgdr.fit(X_norm, y_train)
print(sgdr)
print("Épocas:", sgdr.n_iter_)
print("Contador de atualizações t_:", sgdr.t_)
print("Parâmetros de configuração:", sgdr.get_params())

## 5. Visualizar w e b aprendidos

coef_ contém um peso por característica. intercept_ contém o intercepto como vetor de um elemento. Ambos são atributos aprendidos pelo fit, indicados pelo sufixo _.

O material original compara com pesos normalizados do laboratório anterior: [110.56, -21.27, -32.71, -37.97] e b=363.16. São valores de referência registrados no material para o conjunto de imóveis do curso; não devem ser esperados aqui. Pesos de um modelo normalizado se referem a uma mudança de um desvio padrão de determinada característica, mantendo as outras fixas. Não demonstram causalidade.

A transformação inversa dos pesos, como no Lab 04, permite representar a mesma função em unidades originais:
$$w_j^{original}=w_j^{norm}/\sigma_j,\quad b^{original}=b^{norm}-\vec w^{original}\cdot\vec\mu$$

In [ ]:
w_norm = sgdr.coef_
b_norm = sgdr.intercept_
print("w:", w_norm, "shape:", w_norm.shape)
print("b:", b_norm, "shape:", b_norm.shape)
w_original = w_norm / scaler.scale_
b_original = b_norm[0] - w_original @ scaler.mean_
print("Pesos nas unidades originais:", w_original)
print("Intercepto original:", b_original)

## 6. Previsões: função pronta versus cálculo manual

A previsão continua sendo $\hat y^{(i)}=\vec w\cdot\vec x^{(i)}+b$. Para várias linhas, a notação vetorial é $\hat{\mathbf y}=X_{\rm norm}\vec w+b$.

predict aplica o modelo já treinado. np.dot(X_norm,w_norm) calcula o produto matriz-vetor, resultando em m previsões. Somar o intercepto é broadcasting: o mesmo valor é somado a todos os elementos.

O original compara os resultados com (y_pred == y_pred_sgd).all(). Como floats podem sofrer pequenas diferenças de arredondamento, acrescentamos np.allclose, comparação numérica com tolerância. Os quatro primeiros alvos são y_train[:4]; as previsões não precisam ser iguais aos alvos, pois há ruído e erro residual.

In [ ]:
y_pred_sgd = sgdr.predict(X_norm)
y_pred = np.dot(X_norm, w_norm) + b_norm
print("Igualdade exata:", (y_pred == y_pred_sgd).all())
print("Equivalência numérica:", np.allclose(y_pred, y_pred_sgd))
print("Previsões:", y_pred[:4])
print("Valores reais:", y_train[:4])
print("Previsão na escala original coincide:",
      np.allclose(X_train @ w_original + b_original, y_pred))

## 7. Custo, erro e gráfico de resultados

O original desenha quatro gráficos: preço real e previsto contra cada característica original. A previsão usa as entradas normalizadas; o eixo horizontal usa as unidades reais para facilitar a interpretação. Cada ponto representa uma casa. A combinação das quatro variáveis pode produzir uma nuvem de pontos e não uma reta em cada gráfico isolado.

Como complemento, calculamos o custo quadrático da aula $J=\frac{1}{2m}\sum_i(\hat y^{(i)}-y^{(i)})^2$ e RMSE, raiz do erro quadrático médio. O J abaixo não inclui a penalização L2 interna do SGDRegressor, portanto não é exatamente sua função objetivo regularizada. Ambos são calculados no treino, não medem generalização.

In [ ]:
fig, ax = plt.subplots(1,4,figsize=(12,3),sharey=True)
for j in range(len(ax)):
    ax[j].scatter(X_train[:,j], y_train, label="Real", alpha=.65)
    ax[j].scatter(X_train[:,j], y_pred, label="Previsto", alpha=.65)
    ax[j].set_xlabel(X_features[j])
ax[0].set_ylabel("Preço (mil US$)")
ax[0].legend()
fig.suptitle("Real versus previsto")
plt.tight_layout()
plt.show()
erros = y_pred - y_train
print("Custo J:", np.mean(erros**2)/2)
print("RMSE (mil US$):", np.sqrt(np.mean(erros**2)))

## 8. Prever uma casa nova (complemento didático)

O scaler já aprendeu média e escala das quatro colunas. Use transform, nunca fit_transform, em uma casa nova. O método predict exige uma matriz 2D: [[1200,3,1,40]] possui uma linha e quatro colunas. O retorno é um vetor com uma previsão. Multiplicamos por 1000 para converter milhares de dólares em dólares. A previsão é simulada.

In [ ]:
casa = np.array([[1200,3,1,40]])
casa_norm = scaler.transform(casa)
preco = sgdr.predict(casa_norm)[0]
print("Casa normalizada:", casa_norm)
print(f"Preço previsto: US$ {preco*1000:,.2f}")

## 9. Revisão

- StandardScaler: aprende e aplica Z-score.
- SGDRegressor: treina uma regressão com atualizações estocásticas.
- fit: aprende parâmetros; predict: utiliza parâmetros já aprendidos.
- coef_ e intercept_: pesos e intercepto; max_iter: limite de épocas.
- alpha do SGDRegressor não significa a mesma coisa que alpha da fórmula manual.
- np.dot e predict representam a mesma previsão linear; np.allclose compara floats com tolerância.

**Exercícios:** (1) Por que usar transform na casa nova? (2) Qual a forma de coef_? (3) Qual a diferença entre uma época e uma atualização? (4) Compare SGDRegressor(max_iter=10, random_state=42) com o modelo anterior, usando o mesmo X_norm. (5) Qual a diferença entre custo de treino e generalização?

**Respostas:** (1) Para preservar as estatísticas do treino. (2) (4,). (3) Uma época percorre o conjunto; uma atualização SGD usa um exemplo. (4) Execute e compare, sem supor que mais épocas sempre melhoram dados de teste. (5) Generalização exige dados não usados para aprender ou selecionar parâmetros.

Fonte: laboratório opcional Linear Regression using Scikit-Learn (DeepLearning.AI / Stanford Online), arquivo fornecido pelo aluno.